# Why BERT for News Credibility Analysis

limitations of tf-idf
better understanding of context
subtle language
emotional real news
proffesional fake news




In [10]:
import os

os.environ["HF_HOME"] = "D:/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "D:/hf_cache"
os.environ["TORCH_HOME"] = "D:/torch_cache"
os.environ["TMPDIR"] = "D:/tmp"


In [11]:
import os

print("HF_HOME:", os.environ.get("HF_HOME"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))
print("TORCH_HOME:", os.environ.get("TORCH_HOME"))
print("TMPDIR:", os.environ.get("TMPDIR"))


HF_HOME: D:/hf_cache
TRANSFORMERS_CACHE: D:/hf_cache
TORCH_HOME: D:/torch_cache
TMPDIR: D:/tmp


In [12]:
from transformers.utils import default_cache_path
print("HF default cache path:", default_cache_path)


HF default cache path: D:/hf_cache\hub


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"  # or your actual model

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir="D:/hf_cache"
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    cache_dir="D:/hf_cache"
)

print("✅ Model & tokenizer loaded from D drive")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model & tokenizer loaded from D drive


In [14]:
#!pip install transformers datasets torch scikit-learn

In [15]:
#!pip install torch

In [16]:
# import torch
# print(torch.__version__)


In [17]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)


In [18]:
df = pd.read_csv("../data/combined_news.csv")
df = df[["text", "label"]]   # keep only required columns
df.head()

,text,label
0,"21st Century Wire says Ben Stein, reputable pr...",1
1,WASHINGTON (Reuters) - U.S. President Donald T...,0
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,0
3,"On Monday, Donald Trump once again embarrassed...",1
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",0


In [19]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size = 0.2,
    random_state = 42,
    stratify = df["label"]
)
# ensures the training and testing sets maintain the same proportion of each class (like 0s and 1s, or different categories) as the original dataset

In [20]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [21]:
def tokenize_texts(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256
    )
train_encodings = tokenize_texts(train_texts)
test_encodings = tokenize_texts(test_texts)


In [22]:
import torch

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.labels[idx])
        }


In [23]:
train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

In [24]:
# bert-base-uncased -> pretrained on english understands language structure,grammer
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels = 2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
# !pip install -U transformers


In [26]:
# import transformers
# print(transformers.__version__)


In [27]:
# !pip install -U accelerate


In [28]:
#import sys
#!{sys.executable} -m pip install -U accelerate


In [29]:
#import accelerate
#print(accelerate.__version__)


In [30]:
from transformers import TrainingArguments
# Dataset → Tokenizer → Encodings → Dataset object → Trainer → Training

training_args = TrainingArguments(
    output_dir = "D:/sentiment_output",
    eval_strategy = "epoch",  #after every fullpass training evalute model
    save_strategy = "no",
    logging_dir = "no",
    learning_rate=2e-5,  #fine-tune gently
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,   # Bert converts fast
    weight_decay=0.01,  # regularization -> dont reley too much on any single features
    load_best_model_at_end=False,
    report_to= "none"  
)

In [31]:
from sklearn.metrics import accuracy_score , precision_recall_fscore_support

#
def compute_metrics(eval_pred):
    logits, labels = eval_pred  #logit is raw score and convert them to predictions
    preds = logits.argmax(axis=1) #choose class with highest score

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy":acc,
        "f1":f1,
        "precision": precision,
        "recall":recall
    }

In [32]:
from transformers import Trainer
#Trainer handles training loop, evaluation, logging, saving best model
#Uses eval loss + F1 to select best checkpoint
#Automatically applies backprop, optimizer, scheduler so we can  skip huge coding.

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset= test_dataset,
    tokenizer = tokenizer,
    compute_metrics = compute_metrics
)

C:\Users\HP\AppData\Local\Temp\ipykernel_18740\3549417733.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
#trainer.train()  #dont run it will cost you a lot 

c:\Users\HP\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 